# Circuit Quickstart

A 5-minute tour of the Qumulator circuits API. You will:

1. Submit a **Bell state** circuit and verify the statevector amplitudes
2. Submit a **GHZ state** (N=6) and check the measurement counts
3. Run a **random circuit** in `statevector` mode and inspect the result

All computation happens in Qumulator's cloud engine — no local quantum simulator needed.

---

**Prerequisites:** Install the SDK once:
```bash
pip install qumulator-sdk
```
Then set your API key:
```python
import os
os.environ["QUMULATOR_API_KEY"] = "your_key_here"
```

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import os
import math

from qumulator import QumulatorClient

# Reads QUMULATOR_API_URL and QUMULATOR_API_KEY from environment.
# Default URL is https://api.qumulator.com — set QUMULATOR_API_URL to
# http://localhost:10000 if running a local Docker instance.
client = QumulatorClient()
print(f"API URL : {client.base_url}")
print("Client  : ready")

## Cell 1 — Bell state

Create the Bell state $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$
by applying a Hadamard gate to qubit 0, then a CNOT with qubit 0 as control and qubit 1 as target.

The exact statevector amplitudes should satisfy $|\alpha_{00}| = |\alpha_{11}| = 1/\sqrt{2} \approx 0.7071$.

In [ ]:
# Submit Bell state circuit
result = client.circuits.run(
    n_qubits=2,
    mode="statevector",
    instructions=[
        {"gate": "h",  "qubits": [0]},
        {"gate": "cx", "qubits": [0, 1]},
    ],
    shots=1024,
    return_statevector=True,
)

sv_r = result.statevector_real
sv_i = result.statevector_imag
sv = [complex(r, i) for r, i in zip(sv_r, sv_i)]

print("Bell state amplitudes:")
for k, a in enumerate(sv):
    if abs(a) > 1e-9:
        print(f"  |{k:02b}>  {a:.6f}")

amp_target = 1.0 / math.sqrt(2)
err = max(abs(abs(sv[0]) - amp_target), abs(abs(sv[3]) - amp_target))
print(f"  L∞ error vs 1/√2 = {err:.2e}")
assert err < 1e-10, f"Bell amplitude error too large: {err}"
print("  ✓ Bell state — EXACT")

## Cell 2 — GHZ state (N=6)

The GHZ state $|\text{GHZ}_6\rangle = \frac{1}{\sqrt{2}}(|000000\rangle + |111111\rangle)$
is created by a Hadamard on qubit 0 followed by a cascade of CNOT gates.

After 1024 shots, only the $|000000\rangle$ and $|111111\rangle$ bitstrings should appear.

In [ ]:
n = 6
instructions = [{"gate": "h", "qubits": [0]}]
for q in range(n - 1):
    instructions.append({"gate": "cx", "qubits": [q, q + 1]})

result = client.circuits.run(
    n_qubits=n,
    mode="statevector",
    instructions=instructions,
    shots=1024,
    return_statevector=True,
)

sv = [complex(r, i) for r, i in zip(result.statevector_real, result.statevector_imag)]
nonzero = [(k, a) for k, a in enumerate(sv) if abs(a) > 1e-9]
print(f"GHZ-{n} non-zero amplitudes: {len(nonzero)}  (expect 2)")
for k, a in nonzero:
    print(f"  |{k:0{n}b}>  {a:.6f}")

counts = result.counts
print(f"\nSample counts (top 5): {dict(sorted(counts.items(), key=lambda x: -x[1])[:5])}")

all_zeros = f"{0:0{n}b}"
all_ones  = f"{(1 << n) - 1:0{n}b}"
assert all_zeros in counts and all_ones in counts, "GHZ counts missing expected bitstrings"
assert len(nonzero) == 2, f"Expected 2 non-zero amplitudes, got {len(nonzero)}"
print("  ✓ GHZ state — EXACT")

## Cell 3 — Measurement-only circuit

Demonstrate the `shots`-only mode: submit a circuit with no `return_statevector` flag.
The result contains only measurement counts (no amplitude arrays).

Circuit: random single-qubit rotations on 4 qubits followed by CNOT entangling gates.

In [ ]:
import math

result = client.circuits.run(
    n_qubits=4,
    mode="statevector",
    instructions=[
        {"gate": "ry", "qubits": [0], "params": [math.pi / 3]},
        {"gate": "ry", "qubits": [1], "params": [math.pi / 5]},
        {"gate": "ry", "qubits": [2], "params": [math.pi / 7]},
        {"gate": "ry", "qubits": [3], "params": [math.pi / 4]},
        {"gate": "cx", "qubits": [0, 1]},
        {"gate": "cx", "qubits": [2, 3]},
        {"gate": "cx", "qubits": [1, 2]},
    ],
    shots=2048,
)

counts = result.counts
total_shots = sum(counts.values())
print(f"Total shots recorded: {total_shots}  (expect 2048)")
print(f"Distinct bitstrings: {len(counts)}")
print("Top 5 outcomes:")
for bs, cnt in sorted(counts.items(), key=lambda x: -x[1])[:5]:
    print(f"  |{bs}>  {cnt} shots  ({100*cnt/total_shots:.1f}%)")

assert total_shots == 2048, f"Expected 2048 shots, got {total_shots}"
assert len(counts) > 1, "Expected multiple distinct bitstrings"
print("  ✓ Measurement-only mode works")

## Summary

| Cell | Circuit | Mode | Verified |
|------|---------|------|----------|
| 1 | Bell state (N=2) | `statevector` | Amplitudes exact |
| 2 | GHZ state (N=6) | `statevector` | Only 2 non-zero amplitudes |
| 3 | Mixed rotation + CX (N=4) | `statevector` | Counts sum to 2048 |

**Next steps:** See `cluster_statevector_demo.ipynb` for N=50+ circuits with the cluster engine,
or `h2_ground_state.ipynb` for molecular simulation.